In [1]:
import h5py
import json

import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge, LinearRegression, SGDRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RBF
import time
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import train_test_split

In [2]:
with h5py.File('solutions.h5', 'r') as f:
    # List all groups/datasets
    dataset = f["data"]   # This does NOT load data
    dataset = dataset[:]
    print(dataset.shape)        # Fast
    coef = f["coeffs"]
    coef = coef[:]
    print(coef.shape)

(14000, 64, 128, 1)
(14000, 64)


In [3]:
X_train = dataset[:10000,0,:,0]
Y_train = dataset[:10000,:,:,0]
X_test = dataset[10000:,0,:,0]
Y_test = dataset[10000:,:,:,0]
coef_train = coef[:10000]
coef_test = coef[10000:]
print(X_train.shape, coef_train.shape, Y_train.shape)

(10000, 128) (10000, 64) (10000, 64, 128)


# Vanila Kernel

In [6]:
##################### RBF kernel
bandwidth = 10
kernel = RBF(length_scale = bandwidth)
model = GaussianProcessRegressor(kernel, alpha = 1e-10, optimizer=None)
model.fit(X_train, Y_train.reshape(10000,64*128))

# prediction
pred = model.predict(X_test)
pred = pred.reshape(Y_test.shape)
e = np.mean(np.linalg.norm(pred - Y_test, axis = (1,2))/np.linalg.norm(Y_test, axis = (1,2)))
print(f'Test error is {e:.2e}.\n')

Test error is 5.80e-01.



In [5]:
bandwidth = 100
nu = 3/2
kernel = Matern(length_scale = bandwidth, nu = nu)
model = GaussianProcessRegressor(kernel, alpha = 1e-10, optimizer=None)
model.fit(X_train, Y_train.reshape(10000,64*128))

# prediction
pred = model.predict(X_test)
pred = pred.reshape(Y_test.shape)
e = np.mean(np.linalg.norm(pred - Y_test, axis = (1,2))/np.linalg.norm(Y_test, axis = (1,2)))
print(f'Test error is {e:.2e}.\n')

Test error is 8.39e-01.



# Our Method

In [9]:
# kernel for coefficients
bandwidth1 = 1
nu1 = 5/2
kernel1 = Matern(length_scale = bandwidth1, nu=nu1)
K_c = kernel1(coef_train)

# kernel for u
bandwidth2 = 1000
nu2 = 5/2
kernel2 = Matern(length_scale = bandwidth2, nu=nu2)
K_u = kernel2(X_train)

# product kernel
K_train = K_c * K_u

# model
model = KernelRidge(alpha = 1e-10, kernel='precomputed')
model.fit(K_train, Y_train.reshape(10000,64*128))

# prediction
K_c_test = kernel1(coef_test, coef_train)
K_u_test = kernel2(X_test, X_train)
K_test = K_c_test * K_u_test
our_pred = model.predict(K_test)
our_pred = our_pred.reshape(Y_test.shape)

# Error
our_e = np.mean(np.linalg.norm(our_pred - Y_test, axis = (1,2))/np.linalg.norm(Y_test, axis = (1,2)))
print(f'Test error of our method is {our_e:.2e}.\n')

Test error of our method is 4.88e-02.

